# Building Custom Pipelines with VulnPipeline

`VulnPipeline` is the full ETL orchestrator: extractors run concurrently,
transforms execute in dependency-aware phases, and loaders write the output.
Use it when you need explicit control over every stage.

## Basic Pipeline

Extract the NVD Modified feed, run a CWE → CAPEC → STRIDE chain
plus KEV, and save to Parquet.

In [9]:
from vulnflow import VulnPipeline
from vulnflow.extractors.nvd import NVDExtractor
from vulnflow.helpers import CacheBuilder
from vulnflow.loaders.files import ParquetLoader
from vulnflow.transforms.enrichers.capec import CAPECEnricher
from vulnflow.transforms.enrichers.cwe import CWEEnricher
from vulnflow.transforms.enrichers.epss import EPSSEnricher
from vulnflow.transforms.enrichers.kev import KEVEnricher
from vulnflow.transforms.flatten.capec import CAPECFlattenTransform
from vulnflow.transforms.flatten.cvss import CVSSFlattenTransform
from vulnflow.transforms.flatten.cwe import CWEFlattenTransform
from vulnflow.transforms.flatten.epss import EPSSFlattenTransform
from vulnflow.transforms.flatten.kev import KEVFlattenTransform
from vulnflow.transforms.flatten.stride import STRIDEFlattenTransform
from vulnflow.transforms.inferences.stride import STRIDEInference

pipeline = VulnPipeline(
    extractors=[
        NVDExtractor(CacheBuilder.feed_url("modified")),
    ],
    transformers=[
        # CWE → CAPEC → STRIDE chain
        CWEEnricher(),
        CWEFlattenTransform(),
        CAPECEnricher(),
        CAPECFlattenTransform(),
        CVSSFlattenTransform(),
        STRIDEInference(),
        STRIDEFlattenTransform(),
        # KEV → EPSS → Events → CVDState chain
        KEVEnricher(),
        KEVFlattenTransform(),
        EPSSEnricher(),
        EPSSFlattenTransform(),
    ],
    loaders=[
        ParquetLoader("../data/custom_pipeline.parquet"),
    ],
)

result = pipeline.run()
print(f"{len(result.dataset):,} records")

Processing feeds:   0%|          | 0/1 [00:00]

Enriching & transforming:   0%|          | 0/11 [00:00]

Loading data:   0%|          | 0/1 [00:00<?, ?loaders/s]

2,283 records


In [10]:
print(f"Duration          : {result.duration:.1f}s")
print(f"Successful stages : {result.successful_stages}")
print(f"Failed stages     : {result.failed_stages}")
print(f"Errors            : {result.errors}")
print(f"Loader outputs    : {result.loader_outputs}")

Duration          : 6.0s
Successful stages : ['NVDExtractor(nvdcve-2.0-Modified)', 'CVSSFlattenTransform()', 'EPSSEnricher()', 'CWEEnricher()', 'KEVEnricher()', 'KEVFlattenTransform()', 'CAPECEnricher()', 'CWEFlattenTransform()', 'EPSSFlattenTransform()', 'STRIDEInference()', 'CAPECFlattenTransform()', 'STRIDEFlattenTransform()', 'ParquetLoader(custom_pipeline.parquet)']
Failed stages     : []
Errors            : []
Loader outputs    : {'ParquetLoader(custom_pipeline.parquet)': {'path': '..\\data\\custom_pipeline.parquet', 'record_count': 2283}}


## Transform Dependencies

Transforms declare `depends_on` to control execution ordering.
The pipeline groups them into phases — each phase runs after its
dependencies complete.

In [11]:
transforms = pipeline.transformers

print(f"{'Transform':<30s} {'Depends On'}")
print("-" * 60)
for t in transforms:
    deps = getattr(t, "depends_on", [])
    print(f"{t.name:<30s} {deps if deps else '(none)'}")

Transform                      Depends On
------------------------------------------------------------
CWEEnricher                    (none)
CWEFlattenTransform            ['CWEEnricher']
CAPECEnricher                  ['CWEEnricher']
CAPECFlattenTransform          ['CAPECEnricher']
CVSSFlattenTransform           (none)
STRIDEInference                ['CAPECEnricher', 'CWEFlattenTransform', 'CVSSFlattenTransform']
STRIDEFlattenTransform         ['STRIDEInference']
KEVEnricher                    (none)
KEVFlattenTransform            ['KEVEnricher']
EPSSEnricher                   (none)
EPSSFlattenTransform           ['EPSSEnricher']


In [13]:
from vulnflow.utils.graph import build_phases

phases = build_phases(transforms)

for i, phase in enumerate(phases):
    names = [t.name for t in phase]
    print(f"Phase {i}: {names}")

Phase 0: ['CVSSFlattenTransform', 'EPSSEnricher', 'CWEEnricher', 'KEVEnricher']
Phase 1: ['KEVFlattenTransform', 'CAPECEnricher', 'CWEFlattenTransform', 'EPSSFlattenTransform']
Phase 2: ['STRIDEInference', 'CAPECFlattenTransform']
Phase 3: ['STRIDEFlattenTransform']


## Composition: merge()

`merge()` combines stage lists by type (extractors + extractors,
transforms + transforms, loaders + loaders). Use it to build
reusable pipeline components.

In [15]:
extract = VulnPipeline(
    extractors=[NVDExtractor(CacheBuilder.feed_url("recent"))],
)

enrich = VulnPipeline(
    transformers=[
        KEVEnricher(),
        KEVFlattenTransform(),
        EPSSEnricher(),
        EPSSFlattenTransform(),
        CVSSFlattenTransform(),
    ],
)

load = VulnPipeline(
    loaders=[ParquetLoader("../data/merged_pipeline.parquet")],
)

merged = extract.merge(enrich).merge(load)
result = merged.run()

print(f"{len(result.dataset):,} records, {result.duration:.1f}s")
print(f"Stages: {result.successful_stages}")

Processing feeds:   0%|          | 0/1 [00:00]

Enriching & transforming:   0%|          | 0/5 [00:00]

Loading data:   0%|          | 0/1 [00:00<?, ?loaders/s]

1,361 records, 5.6s
Stages: ['NVDExtractor(nvdcve-2.0-Recent)', 'EPSSEnricher()', 'CVSSFlattenTransform()', 'KEVEnricher()', 'KEVFlattenTransform()', 'EPSSFlattenTransform()', 'ParquetLoader(merged_pipeline.parquet)']


## Composition: then()

`then()` chains **transform-only** pipelines. It validates that neither
pipeline has extractors or loaders — use it for building reusable
transform chains.

In [17]:
enrichment_chain = VulnPipeline(
    transformers=[KEVEnricher(), KEVFlattenTransform()],
)

scoring_chain = VulnPipeline(
    transformers=[EPSSEnricher(), EPSSFlattenTransform(), CVSSFlattenTransform()],
)

full_transform = enrichment_chain.then(scoring_chain)
print(f"Combined transforms: {[t.name for t in full_transform.transformers]}")

Combined transforms: ['KEVEnricher', 'KEVFlattenTransform', 'EPSSEnricher', 'EPSSFlattenTransform', 'CVSSFlattenTransform']


In [18]:
# then() rejects pipelines with extractors or loaders
try:
    has_extractor = VulnPipeline(
        extractors=[NVDExtractor(CacheBuilder.feed_url("modified"))],
        transformers=[KEVEnricher()],
    )
    has_extractor.then(scoring_chain)
except ValueError as e:
    print(f"ValueError: {e}")

ValueError: then() requires transform-only pipelines but self has extractors. Use merge() for general stage combining.
